# Generating human annotation results

This notebook compares human and AI feedback forensics annotations 

In [ ]:
import json
from feedback_forensics.data.operations.core import load_ap, save_ap
from inverse_cai.data.annotated_pairs_format import hash_string

# load human annotations for personality traits

ap = load_ap("/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json")

# filter out comparisons with issues
# (by selecting comparisons where issue annotator is irrelevant)
ap["annotators"]
issue_language_annotator = "9ef9bb2f"
filtered_comparisons = []
for comp in ap["comparisons"]:
    if comp["annotations"].get(issue_language_annotator, {}).get("pref") == "irrelevant":
        filtered_comparisons.append(comp)
print(f"Filtered down to {len(filtered_comparisons)} comparisons")
# Limit to 100 comparisons
ap["comparisons"] = filtered_comparisons[:min(100, len(filtered_comparisons))]
print(f"Final number of comparisons: {len(ap['comparisons'])}")

In [ ]:
### To run these experiments use:
# ff-annotate -d "/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json" -m "openrouter/openai/gpt-oss-120b"
# ff-annotate -d "/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json" -m "openrouter/openai/gpt-5-mini"
# ff-annotate -d "/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json" -m "openrouter/google/gemini-2.5-flash"
# icai-exp data_path="/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_more_langproblem_ann.json" annotator.skip=true s0_skip_principle_generation=true alg_model="openrouter/google/gemini-2.5-flash" "s0_added_principles_to_test=[\"Select the response that is more concise\",\"Select the response that is more verbose\",\"Select the response that provides a numbered list format\",\"Select the response that has more structured formatting\",\"Select the response that ends with a follow-up question\",\"Select the response that more strictly follows the requested output format\",\"Select the response that is more polite\",\"Select the response that has a friendlier tone\",\"Select the response that uses more casual language\",\"Select the response that uses more formal language\",\"Select the response that includes inappropriate language\",\"Select the response that suggests illegal activities\",\"Select the response that has a more avoidant tone\",\"Select the response that is more factually correct\",\"Select the response that is more offensive\",\"Select the response that includes more references to other sources\",\"Select the response that expresses more emotion\",\"Select the response that contains less harmful information\",\"Select the response that refuses to answer the question\",\"Select the response that uses more bold and italics text\",\"Select the response that provides more examples\",\"Select the response that uses more humour\",\"Select the response that uses more personal pronouns (I, we, you)\",\"Select the response that includes more ethical considerations\",\"Select the response that acknowledges own limitations or uncertainty more\",\"Select the response that is more creative and original\",\"Select the response that makes more confident statements\",\"Select the response that provides conclusions without full reasoning\",\"Select the response that actively engages the reader with rhetorical questions\",\"Select the response that uses a more enthusiastic tone\",\"Select the response that uses more mathematical symbols and notation\",\"Select the response that uses more emojis\",\"Select the response that compliments the user's question or prompt\",\"Select the response that agrees more with the user\",\"Select the response that agrees with user even if factually incorrect\",\"Select the response that reinforces user's beliefs more\",\"Select the response that reinforces user's anger more\",\"Select the response that is more empathetic to the user\",\"Select the response that is more optimistic\",\"Select the response that more actively engages with the user\"]" s3_num_seeds_to_reannotate_with=1

other_data = {
    "gpt4o-mini": "/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena.json",
    "gpt-5-mini": "/Users/arduin/main/repos/github/feedback-forensics/exp/outputs/2025-09-23_19-10-04_gpt-5-mini/results/070_annotations_train_ap.json",
    #"gpt-oss-120b": "/Users/arduin/main/repos/github/feedback-forensics/exp/outputs/2025-08-23_21-49-44_oss/results/070_annotations_train_ap.json",
    "gemini-flash-2.5": "/Users/arduin/main/repos/github/feedback-forensics/exp/outputs/2025-08-23_22-30-59_geminiflash25/results/070_annotations_train_ap.json",
    # "gemini-flash-2.5-v2": "/Users/arduin/main/repos/github/feedback-forensics/exp/outputs/2025-09-23_19-36-19_geminiflash25_v2/results/070_annotations_train_ap.json",
    "gemini-flash-2.5-single": "/Users/arduin/main/repos/github/feedback-forensics/exp/outputs/2025-09-23_19-26-17_geminiflash25_single/results/070_annotations_train_ap.json",
}

human_principle_annotators_descriptions = [
    annotator["description"].replace("Human:", "").strip() for annotator in ap["annotators"].values() if "human" in annotator["description"].lower()
]

def get_comparison(id: str, ap: dict):
    for comparison in ap["comparisons"]:
        if comparison["id"] == id:
            return comparison

def add_other_data(ap: dict, other_data: dict):
    for annotator_name, path in other_data.items():
        print(f"Adding {annotator_name} data from {path}")
        new_ap = load_ap(path)

        # Get overlapping annotators and update their description and hashes
        overlapping_annotators = {
            a_hash: annotator for a_hash, annotator in new_ap["annotators"].items() if annotator["description"].replace("Select the response that ", "").strip() in human_principle_annotators_descriptions
        }
        print(f"Overlapping annotators:\n\n{"\n".join([ann['description'] for ann in overlapping_annotators.values()])}.")

        for annotator in overlapping_annotators.values():
            annotator["description"] = annotator_name + ": " + annotator["description"].replace("Select the response that ", "")

        hash_conversion = {
            old_hash: hash_string(annotator["description"]) for old_hash, annotator in overlapping_annotators.items()
        }
        overlapping_annotators = {
            hash_conversion[old_hash]: annotator for old_hash, annotator in overlapping_annotators.items()
        }

        # Add overlapping annotators to main AP
        ap["annotators"].update(overlapping_annotators)

        for comparison in ap["comparisons"]:
            new_comparison = get_comparison(comparison["id"], new_ap)
            for old_hash, new_hash in hash_conversion.items():
                comparison["annotations"][new_hash] = new_comparison["annotations"][old_hash]

    return ap

ap = add_other_data(ap=ap, other_data=other_data)

# save combined ap to file
save_ap(ap, file_path="/Users/arduin/main/repos/huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_combined_v3.json")

In [ ]:
# Load combined AP with FF handler for metric analysis

import feedback_forensics as ff
import pathlib

dataset = ff.DatasetHandler()
data_path = pathlib.Path("../../../huggingface/feedback-forensics-public-results/paper/rebuttal-results/chatbot_arena_human_ap_v5_combined_v3.json")
dataset.add_data_from_path(data_path)
df = dataset.first_handler.df

In [ ]:
df.to_json("test.json")

In [ ]:
annotator_metadata = dataset.get_available_annotators()
def get_annotator_key(in_row_name: str) -> str:
    annotator_keys = []
    for annotator_key, metadata in annotator_metadata.items():
        if metadata["annotator_in_row_name"] == in_row_name:
            annotator_keys.append(annotator_key)

    assert len(annotator_keys) == 1, f"None or multiple annotator keys found for {in_row_name}: {annotator_keys}"
    return annotator_keys[0]

In [ ]:
import sklearn.metrics

relevant_principles = [
 'is more verbose',
 'has more structured formatting',
 'makes more confident statements',
 'is more factually correct',
 'more strictly follows the requested output format',
 'is more concise',
 'has a more avoidant tone',
 'refuses to answer the question',
 'ends with a follow-up question',
 'is more polite',
]

annotator_overall_data = {}
for annotator_name in other_data.keys():
    annotator_data = {}
    for principle in relevant_principles:
        annotator_data[principle] = {}
        llm_hash = get_annotator_key(f"{annotator_name}: {principle}")
        human_hash = get_annotator_key(f"Human: {principle}")

        llm_data = df[llm_hash]
        human_data = df[human_hash]

        annotator_data[principle]["kappa"] = sklearn.metrics.cohen_kappa_score(
            df[llm_hash].to_numpy(dtype="str"),
            df[human_hash].to_numpy(dtype="str"),
        )
        annotator_data[principle]["agreement"] = sklearn.metrics.accuracy_score(
            df[llm_hash].to_numpy(dtype="str"),
            df[human_hash].to_numpy(dtype="str"),
        )

        relevant_values = ["text_a", "text_b"]

        df[f"{llm_hash}_relevant"] = df[llm_hash].isin(relevant_values)
        df[f"{human_hash}_relevant"] = df[human_hash].isin(relevant_values)

        annotator_data[principle]["agreement_on_relevance"] = sklearn.metrics.accuracy_score(
            df[f"{llm_hash}_relevant"].to_numpy(dtype="str"),
            df[f"{human_hash}_relevant"].to_numpy(dtype="str"),
        )

        relevant_values = ["text_a", "text_b"]
        relevant_df = df[df[llm_hash].isin(relevant_values) & df[human_hash].isin(relevant_values)]

        annotator_data[principle]["kappa_relevant"] = sklearn.metrics.cohen_kappa_score(
            relevant_df[llm_hash].to_numpy(dtype="str"),
            relevant_df[human_hash].to_numpy(dtype="str"),
        )
        annotator_data[principle]["agreement_relevant"] = sklearn.metrics.accuracy_score(
            relevant_df[llm_hash].to_numpy(dtype="str"),
            relevant_df[human_hash].to_numpy(dtype="str"),
        )
        annotator_data[principle]["prop_both_rel"] = len(relevant_df) / len(df)
        annotator_data[principle]["prop_invalid"] = len(df[df[llm_hash] == "invalid"]) / len(df)

    annotator_overall_data[annotator_name] = annotator_data


In [ ]:
import pandas as pd

d = annotator_overall_data
cols = ['agreement_on_relevance', 'agreement_relevant'] #'prop_invalid']

table_df = pd.concat(
    {model: pd.DataFrame(traits).T[cols] for model, traits in d.items()},
    axis=1
)

idx = pd.IndexSlice
styler = table_df.style
for metric in cols:
    styler = styler.highlight_max(axis=1, subset=idx[:, idx[:, metric]], props='font-color: white; background-color: blue')

styler = styler.format(precision=2, na_rep='—')
styler